## Unity Catalog Volume JSON → Delta
Point this notebook at a Unity Catalog volume that stores JSON objects. It will ingest the files, add basic bronze metadata, and persist the result as a managed Delta table.

### Notebook widgets

In [0]:
dbutils.widgets.text("volume_catalog", "main", "Volume Catalog")
dbutils.widgets.text("volume_schema", "skew_demo", "Volume Schema")
dbutils.widgets.text("volume_name", "landing", "Volume Name")
dbutils.widgets.text("relative_path", "datasets/json/", "Folder inside Volume")
dbutils.widgets.text("target_catalog", "main", "Target Catalog")
dbutils.widgets.text("target_schema", "skew_demo", "Target Schema")
dbutils.widgets.text("target_table", "volume_json_bronze", "Delta Table Name")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "Delta Write Mode")

### Imports & Spark config

In [0]:
from pyspark.sql import functions as F

# spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
# spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

### Resolve paths

In [0]:
volume_catalog = dbutils.widgets.get("volume_catalog")
volume_schema = dbutils.widgets.get("volume_schema")
volume_name = dbutils.widgets.get("volume_name")
relative_path = dbutils.widgets.get("relative_path").strip("/")

if not all([volume_catalog, volume_schema, volume_name]):
    raise ValueError("Volume catalog, schema, and name must be provided")

source_root = f"/Volumes/{volume_catalog}/{volume_schema}/{volume_name}"
source_path = f"{source_root}/{relative_path}" if relative_path else source_root

### Load JSON files from the volume

In [0]:
raw_df = spark.read.format("json").load(source_path)

# Add _metadata.file_path for Unity Catalog compatibility
raw_df = raw_df.withColumn("_metadata_file_path", F.col("_metadata.file_path"))

if raw_df.limit(1).count() == 0:
    raise FileNotFoundError(f"No JSON files found under {source_path}")

display(raw_df.limit(10))

### Bronze conditioning & metadata

In [0]:
bronze_df = (
    raw_df.withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
    .withColumn("_source_volume", F.lit(source_root))
    .withColumn("_source_path", F.col("_metadata_file_path"))
)

### Write to Delta

In [0]:
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")
write_mode = dbutils.widgets.get("write_mode")

if not all([target_catalog, target_schema, target_table]):
    raise ValueError("Target catalog, schema, and table must be provided")

target_fqn = f"`{target_catalog}`.`{target_schema}`.`{target_table}`"

(
    bronze_df.write.format("delta")
    .mode(write_mode)
    .option("mergeSchema", "true")
    .saveAsTable(target_fqn)
)

### Quick profile of the Delta table

In [0]:
display(spark.table(target_fqn).groupBy("_metadata_file_path").count().orderBy(F.desc("count")))